# Context Managers

> Generated from [`06-Context-Managers.py`](06-Context-Managers.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
The `with` statement: how it guarantees setup/cleanup around a block of code (even on error), how to write your own context managers as classes or generators, and handling a variable number of them at once.

### Key points
- A context manager sets up a resource, hands control to a block, then guarantees cleanup afterward — used via `with`. `open()` is the classic built-in example.
- `with` replaces a manual `try/finally` — same guarantee, less error-prone, less boilerplate.
- Class-based context manager: implement `__enter__` (setup, returns the resource) and `__exit__(exc_type, exc_value, traceback)` (cleanup). `__exit__` returning `True` suppresses the exception; `False` (or `None`) lets it propagate.
- Function-based context manager: `@contextlib.contextmanager` on a generator function — code before `yield` is the `__enter__` half, code after (typically in a `finally`) is the `__exit__` half.
- One `with` statement can open multiple managers: `with open(a) as f1, open(b) as f2:`.
- `contextlib.ExitStack` handles a number of context managers decided at **runtime** (e.g. one file per item in a list) — enter each via `stack.enter_context()`, and they all exit in reverse order when the `with` block ends, even on error.
- `contextlib.suppress(SomeError)` is itself a context manager that turns a specific exception into a no-op.
- Async context managers use `__aenter__`/`__aexit__` (both coroutines) with `async with` — full depth in the Async-Await file.

### Gotchas (the part worth revising twice)
- **Reusable vs reentrant**: most `@contextmanager`-based generators are only *reusable* (same instance, separate `with` blocks one after another) but NOT *reentrant* (same instance nested inside itself). Nesting the same `@contextmanager` instance reuses the already-exhausted generator and raises an error.
- `__exit__`'s return value matters: returning `True` silently swallows the exception — easy to accidentally suppress errors you meant to let propagate if you're not careful with the return logic.
- Class-based context managers give you more control (state across `__enter__`/`__exit__`, reentrancy if you design for it); generator-based ones are shorter to write but inherit the reusable-not-reentrant limitation above.

### When it matters
Anywhere you need guaranteed cleanup — files, locks, DB connections, network sockets — especially when the cleanup must happen even if the code in between raises an exception.

### One-line recap example
```python
from contextlib import contextmanager

@contextmanager
def managed_resource(name):
    print(f"acquire {name}")
    try:
        yield name
    finally:
        print(f"release {name}")   # always runs, even on exception
```

In [ ]:
from contextlib import contextmanager

## 01. Context Managers - Introduction

- A context manager sets up a resource, hands control to a block of code,
  then guarantees cleanup afterwards - even if an exception occurs.
- Used via the `with` statement.

## 02. The `with` Statement (built-in example: file handling)

In [ ]:
with open("temp_demo.txt", "w") as f:
    f.write("Hello, context managers!")
print("\n# 02. The `with` Statement")
print("file written and automatically closed")
print("file closed?", f.closed)  # True - closed automatically, even on error

## 03. Why Context Managers? (vs manual try/finally)

In [ ]:
file = open("temp_demo.txt", "r")
try:
    content = file.read()
    print("content:", content)
finally:
    file.close()
print("# - `with` does this automatically and is less error-prone.")

## 04. Writing a Class-Based Context Manager

- Implement __enter__ (setup, returns resource) and __exit__ (cleanup).

In [ ]:
class ManagedFile:
    def __init__(self, filename, mode):
        self.filename = filename
        self.mode = mode

    def __enter__(self):
        print(f"Opening {self.filename}")
        self.file = open(self.filename, self.mode)
        return self.file

    def __exit__(self, exc_type, exc_value, traceback):
        print(f"Closing {self.filename}")
        self.file.close()
        return False  # False -> don't suppress exceptions

print("\n# 04. Writing a Class-Based Context Manager")
with ManagedFile("temp_demo.txt", "r") as f:
    print("read:", f.read())

## 05. Handling Exceptions Inside __exit__

In [ ]:
class SuppressError:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is ZeroDivisionError:
            print(f"Suppressed: {exc_value}")
            return True  # True -> exception is suppressed, code continues
        return False

print("\n# 05. Handling Exceptions Inside __exit__")
with SuppressError():
    result = 10 / 0
print("continues normally after suppressed exception")

## 06. Function-Based Context Manager (contextlib.contextmanager)

In [ ]:
@contextmanager
def managed_resource(name):
    print(f"Acquiring resource: {name}")
    try:
        yield name  # value returned to the `as` variable
    finally:
        print(f"Releasing resource: {name}")

print("\n# 06. Function-Based Context Manager")
with managed_resource("Database Connection") as res:
    print(f"Using {res}")

## 07. Multiple Context Managers in One `with`

In [ ]:
with open("temp_demo.txt", "r") as f1, open("temp_demo.txt", "r") as f2:
    print("both files open:", not f1.closed and not f2.closed)

## 09. contextlib.ExitStack - a Variable Number of Context Managers

- Section 07 works when you know the exact managers ahead of time.
  ExitStack handles a number decided AT RUNTIME (e.g. one file per item
  in a list) - each is entered via stack.enter_context() and all get
  exited in reverse order when the `with` block ends, even on error.

In [ ]:
from contextlib import ExitStack

print("\n# 09. contextlib.ExitStack for a Variable Number of Managers")
filenames = ["temp_a.txt", "temp_b.txt", "temp_c.txt"]
for name in filenames:
    with open(name, "w") as f:
        f.write(f"content of {name}")

with ExitStack() as stack:
    open_files = [stack.enter_context(open(name, "r")) for name in filenames]
    print("all opened at once, count:", len(open_files))
    print("contents:", [f.read() for f in open_files])
print("all files closed automatically when ExitStack's `with` block ends")

## 10. Reentrant vs Reusable Context Managers

- Reusable: the SAME instance can be used in separate `with` blocks,
  one after another (not nested).
- Reentrant: the same instance can be used in NESTED `with` blocks too.
- Most @contextmanager-based generators are only reusable, NOT reentrant -
  entering a second time before the first `with` exits reuses the same
  exhausted generator and raises an error.

In [ ]:
resource = managed_resource("Reusable Demo")
with resource as r1:
    print("first use:", r1)
# using it again AFTER the first block closed - this is fine (reusable)
resource2 = managed_resource("Reusable Demo 2")
with resource2 as r2:
    print("second, separate use:", r2)

try:
    reentrant_attempt = managed_resource("Reentrant Attempt")
    with reentrant_attempt as outer:
        with reentrant_attempt as inner:  # nesting the SAME instance
            print("this line won't be reached")
except Exception as e:
    print(f"nesting the same @contextmanager instance fails: {type(e).__name__}: {e}")

## 11. contextlib.suppress (cross-reference)

- contextlib.suppress(SomeError) is itself a context manager that turns
  a specific exception into a no-op - a cleaner alternative to
  try/except/pass. Full depth is in 03-Error-And-Exception-Handling.py.

In [ ]:
from contextlib import suppress

print("\n# 11. contextlib.suppress (cross-ref to Error Handling file)")
with suppress(FileNotFoundError):
    open("does_not_exist.txt", "r")
print("missing-file error suppressed, execution continues")

## 12. Async Context Managers (brief mention)

- `async with` uses __aenter__/__aexit__ instead of __enter__/__exit__ -
  both are coroutines, awaited during entry/exit (e.g. for an async DB
  connection). Full depth, including a runnable example, is in
  10-Async-Await.py.

## 13. Cleanup

In [ ]:
import os
os.remove("temp_demo.txt")
for name in filenames:
    os.remove(name)
print("\n# 13. Cleanup")
print("temp files removed")